In [0]:
dbutils.library.restartPython()

In [0]:
import random
import time
from datetime import datetime

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t

from quality.quality import Quality

from src.utils.file import FileReader as fr
from src.utils.validations import *
from file_manager.move import FileManager as fm

In [0]:
data_hoje = datetime.now().strftime("%Y%m%d")
dir_path = r"/Volumes/workspace/callcenter/disc_volumes/landing/"
path_processing = r""

In [0]:
arquivos = dbutils.fs.ls(dir_path)

for arquivo in arquivos:

    nm_arquivo = arquivo.name

    if not nm_arquivo.startswith(f"discagem_{data_hoje}_"):
        continue

    try:

        # =====================================
        # LEITURA
        # =====================================
        df = fr.read_csv(
            path=arquivo.path,
            header=True,
            use_spark=True,
            sep=";"
        )

        # =====================================
        # QUALITY
        # =====================================
        valido = Quality.validar_arquivo(
            df=df,
            nm_arquivo=nm_arquivo
        )

        # =====================================
        # APROVADO
        # =====================================
        if valido:

            fm.move_to_approved(
                source_path=arquivo.path
            )

        # =====================================
        # REPROVADO
        # =====================================
        else:

            fm.move_to_rejected(
                source_path=arquivo.path
            )

    except Exception as e:

        fm.move_to_rejected(
            source_path=arquivo.path
        )



In [0]:

spark.sql("select * from quality.execucao_arquivo").display()
spark.sql("select * from quality.erros_linha").display()
spark.sql("select * from quality.erros_arquivo").display()




In [0]:

run_delete = False

if run_delete:
    spark.sql("delete from quality.execucao_arquivo").display()
    spark.sql("delete from quality.erros_linha").display()
    spark.sql("delete from quality.erros_arquivo").display()

